# 04 - PostGIS Geoanalytics

## Goal

Use PostGIS to analyze concert locations and build the geographic logic needed for GigRoute Europe.

## Tasks

- Explore event distribution by city
- Calculate distances from a reference location
- Find concerts within a travel radius
- Find the nearest upcoming concerts
- Validate geographic query results

In [4]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import URL, create_engine, text

In [6]:
project_path = Path("..")

load_dotenv(project_path / ".env", override= True)
db_user = os.getenv("POSTGRES_USER")
db_password = os.getenv("POSTGRES_PASSWORD")
db_name = os.getenv("POSTGRES_DB")
db_port = os.getenv("POSTGRES_PORT")

In [7]:
assert db_user is not None
assert db_password is not None
assert db_name is not None
assert db_port is not None

In [9]:
database_url = URL.create(
    drivername= "postgresql+psycopg2",
    username= db_user,
    password= db_password,
    host= "localhost",
    port= int(db_port),
    database= db_name
)

engine= create_engine(database_url)

In [11]:
with engine.connect() as connection:
    result = connection.execute(
        text(
            "SELECT 1;"
        )
    ).scalar()

print(result)

1


## Event Distribution

The geographic distribution of events is explored before performing distance-based analysis.

In [ ]:
city_counts = pd.read_sql(
    """
    SELECT
        city,
        COUNT(*) AS event_count
    FROM events
    GROUP BY city
    ORDER BY event_count DESC;
    """,
    engine
)

city_counts.head()

,city,event_count
0,Berlin,318
1,Hamburg,179
2,Cologne,166
3,Munich,104
4,Frankfurt am Main,78


In [16]:
city_counts.shape
city_counts["event_count"].sum()

np.int64(1172)

In [18]:
user_latitude = 52.5200
user_longitude = 13.4050

radius_km = 100

## Concerts Within a Travel Radius

PostGIS `ST_DWithin` is used to identify events located within a specified distance from a reference point.

In [19]:
nearby_events_query = text("""
    SELECT
        event_id,
        event_name,
        artist_name,
        event_date,
        venue_name,
        city,
        latitude,
        longitude,
        ST_Distance(
            location,
            ST_SetSRID(
                ST_MakePoint(:user_longitude, :user_latitude),
                4326
            )::geography
        ) / 1000 AS distance_km
    FROM events
    WHERE ST_DWithin(
        location,
        ST_SetSRID(
            ST_MakePoint(:user_longitude, :user_latitude),
            4326
        )::geography,
        :radius_meters
    )
    ORDER BY distance_km;
""")

In [ ]:
nearby_events= pd.read_sql(
    nearby_events_query,
    engine,
    params={
        "user_longitude": user_longitude,
        "user_latitude": user_latitude,
        "radius_meters": radius_km * 1000
    }
)

pandas.DataFrame

In [22]:
print("Events within radius:", len(nearby_events))

Events within radius: 317


In [24]:
nearby_events["distance_km"] = (
    nearby_events["distance_km"]
    .round(1)
)

In [25]:
nearby_events[
    ["event_name", "city", "event_date", "distance_km"]
].head(20)

,event_name,city,event_date,distance_km
0,Curtis On Tour,Berlin,2026-10-09,1.1
1,Ed O’Brien - An Evening With… Blue Morpho Tour,Berlin,2026-10-10,1.1
2,Jacob Collier - The Light For Days Tour,Berlin,2026-09-10,1.1
3,KILIMANJARO,Berlin,2026-11-11,1.4
4,greek - greek! Live,Berlin,2026-11-26,1.4
5,LOUKEMAN,Berlin,2026-10-29,1.4
6,EST Gee - BIGGER THAN THE DEVIL TOUR,Berlin,2026-10-16,1.7
7,HONNE - 10 Year anniversary tour,Berlin,2026-10-13,1.7
8,54 Ultra - LIVE IN EU,Berlin,2026-10-15,1.7
9,zeyne - AWDA EU Tour 2026,Berlin,2026-09-11,1.7
